# Visual wrappers

> Perturb the agents' views (noise, jitter, blur, occlusions, chroma key, ...), as stable-worldmodel's visual wrappers.

In [ ]:
#| default_exp wrappers.visual

In [ ]:
#| hide
from nbdev.showdoc import *

Each wrapper transforms images with an OpenCV / numpy function of one `(H, W, C)` frame:

* the env's `render()`;
* used **after** `MegaWrapper` (`World(extra_wrappers=[...])`): the info's `pixels` (each agent's view,
  `(num_agents, H, W, C)`, transformed agent by agent), `pixels.*` and `render`;
* used **before** it (`World(pre_wrappers=[...])`), where there is no info dict yet: each agent's
  `pixels` observation, from which the info is built.

Random wrappers draw one sample per frame (so each agent's view gets its own noise, shift or cutout),
except the ones fixed for an episode (jitter factors, occlusion patches, conv weights), shared by
the agents. Patches are kept per frame size, so views of different sizes (`pixels` and `render`)
get patches of their own size. Schedules (`linear`, `cosine`, ...) make a parameter depend on the
step count (e.g. `NoiseWrapper(env, std=linear(0, 30, 1000))`).

In [ ]:
#| export
# Adapted from stable-worldmodel's wrappers and plotting utilities (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import math
import pathlib

import cv2
import gymnasium as gym
import numpy as np

VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.webm', '.gif', '.mkv'}


def _load_media(path):
    import imageio.v3 as iio
    path = pathlib.Path(path)
    if path.suffix.lower() in VIDEO_EXTS:
        return np.stack(list(iio.imiter(path)))
    return np.asarray(iio.imread(path))

### Schedules

In [ ]:
#| export
def constant(value):
    "Schedule that always returns `value`."
    return lambda step: value


def linear(start, end, horizon):
    "Linear ramp from `start` to `end` over `horizon` steps; held at `end` after."
    return lambda step: start + (end - start) * min(step / max(1, horizon), 1.0)


def cosine(start, end, horizon):
    "Cosine ramp from `start` to `end` over `horizon` steps; held at `end` after."
    def f(step):
        t = min(step / max(1, horizon), 1.0)
        return end + 0.5 * (start - end) * (1 + math.cos(math.pi * t))
    return f


def exponential(start, decay, floor=0.0):
    "Exponential decay `start * decay**step`, lower-bounded by `floor`."
    return lambda step: max(start * (decay ** step), floor)


def sinusoidal(low, high, period):
    "Sinusoid oscillating in `[low, high]` with the given `period` (in steps)."
    amp, mid = 0.5 * (high - low), 0.5 * (high + low)
    return lambda step: mid + amp * math.sin(2 * math.pi * step / max(1.0, period))

### Base

In [ ]:
#| export
def _is_view_key(k) -> bool:
    return isinstance(k, str) and (k in ('pixels', 'render') or k.startswith('pixels.'))


class _PixelTransform(gym.Wrapper):
    "Applies `_apply` (one `(H, W, C)` frame) to `render()` and to the views (see above)."

    def _apply(self, frame):
        raise NotImplementedError

    def _apply_frames(self, value):
        "One frame `(H, W, C)`, or one per agent `(num_agents, H, W, C)`."
        value = np.asarray(value)
        if value.ndim == 4:
            return np.stack([self._apply(f) for f in value])
        return self._apply(value)

    def _apply_to_info(self, info):
        for k, v in info.items():
            if _is_view_key(k):
                info[k] = self._apply_frames(v)
        return info

    def _apply_to_obs(self, obs):
        "Each agent's `pixels` observation (before MegaWrapper, where there is no info dict yet)."
        if isinstance(obs, dict) and obs and all(isinstance(o, dict) for o in obs.values()):
            return {a: ({**o, 'pixels': self._apply_frames(o['pixels'])} if 'pixels' in o else o) for a, o in obs.items()}
        return obs

    def _transform(self, obs, info):
        if isinstance(info, dict) and any(_is_view_key(k) for k in info):   # after MegaWrapper: the views are in the info
            return obs, self._apply_to_info(info)
        return self._apply_to_obs(obs), info

    def render(self):
        frame = self.env.render()
        return frame if frame is None else self._apply(frame)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return self._transform(obs, info)

    def step(self, action):
        obs, reward, term, trunc, info = self.env.step(action)
        obs, info = self._transform(obs, info)
        return obs, reward, term, trunc, info

### Wrappers

In [ ]:
#| export
class ChromaKeyWrapper(_PixelTransform):
    """
    Replace the pixels matching a key color (within `tolerance`) with an image or video background,
    like a green screen. A video background advances (and loops) once per transformed frame
    batch: the agents' views of a step share the same background frame.
    """
    def __init__(self, env, key_color, media, tolerance=0.0):
        super().__init__(env)
        self._keys = np.atleast_2d(np.asarray(key_color, dtype=np.int16)).reshape(-1, 3)
        self._tol = float(tolerance)
        media = _load_media(media) if isinstance(media, (str, pathlib.Path)) else np.asarray(media)
        self._is_video, self._media, self._idx = media.ndim == 4, media, 0
        self._background = None

    def _next_frame(self, h, w):
        frame = self._media[self._idx] if self._is_video else self._media
        if frame.shape[:2] != (h, w):
            frame = cv2.resize(frame, (w, h), interpolation=cv2.INTER_AREA)
        if self._is_video:
            self._idx = (self._idx + 1) % len(self._media)
        return frame

    def _apply(self, frame):
        h, w = frame.shape[:2]
        background = self._background if self._background is not None and self._background.shape[:2] == (h, w) \
            else self._next_frame(h, w)
        diff = frame.astype(np.int16) - self._keys[:, None, None, :]
        mask = (np.linalg.norm(diff, axis=-1) <= self._tol).any(axis=0)
        out = frame.copy()
        out[mask] = background[mask]
        return out

    def _apply_frames(self, value):
        value = np.asarray(value)
        if value.ndim == 4:                               # one background frame for every agent's view
            self._background = self._next_frame(*value.shape[1:3])
            try:
                return np.stack([self._apply(f) for f in value])
            finally:
                self._background = None
        return self._apply(value)


class NoiseWrapper(_PixelTransform):
    """
    Gaussian pixel noise with a step-dependent standard deviation: `std` is a float or a schedule
    `f(step) -> float`, evaluated at the number of `env.step` calls so far.
    """
    def __init__(self, env, std=10.0, seed=None):
        super().__init__(env)
        self._std = std if callable(std) else constant(std)
        self._rng = np.random.default_rng(seed)
        self._step = 0

    @property
    def step_count(self):
        "Number of `env.step` calls seen (the value fed to the schedule)."
        return self._step

    def _apply(self, frame):
        s = float(self._std(self._step))
        if s <= 0:
            return frame
        noise = self._rng.normal(0, s, frame.shape)
        return np.clip(frame.astype(np.float32) + noise, 0, 255).astype(frame.dtype)

    def step(self, action):
        out = super().step(action)
        self._step += 1
        return out


class ColorJitterWrapper(_PixelTransform):
    "Random brightness, contrast, saturation and hue shifts; factors resampled at each reset."
    def __init__(self, env, brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, seed=None):
        super().__init__(env)
        self._b, self._c, self._s, self._h = float(brightness), float(contrast), float(saturation), float(hue)
        self._rng = np.random.default_rng(seed)
        self._sample()

    def _sample(self):
        self._db = self._rng.uniform(max(0.0, 1 - self._b), 1 + self._b)
        self._dc = self._rng.uniform(max(0.0, 1 - self._c), 1 + self._c)
        self._ds = self._rng.uniform(max(0.0, 1 - self._s), 1 + self._s)
        self._dh = self._rng.uniform(-self._h, self._h)

    def _apply(self, frame):
        f = frame.astype(np.float32) * self._db
        mean = f.mean(axis=(0, 1), keepdims=True)
        f = (f - mean) * self._dc + mean
        rgb = np.clip(f, 0, 255).astype(np.uint8)
        hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[..., 0] = (hsv[..., 0] + self._dh * 180.0) % 180.0
        hsv[..., 1] = np.clip(hsv[..., 1] * self._ds, 0, 255)
        return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB).astype(frame.dtype)

    def reset(self, **kwargs):
        self._sample()
        return super().reset(**kwargs)


class BlurWrapper(_PixelTransform):
    "Gaussian blur with odd `kernel` size and standard deviation `sigma` (0: derived from the kernel)."
    def __init__(self, env, kernel=5, sigma=0.0):
        super().__init__(env)
        self._kernel, self._sigma = int(kernel) | 1, float(sigma)

    def _apply(self, frame):
        return cv2.GaussianBlur(frame, (self._kernel, self._kernel), self._sigma)


class OcclusionWrapper(_PixelTransform):
    "Cover the frame with `num_patches` rectangles of fractional `size`; patches resampled at each reset."
    def __init__(self, env, num_patches=1, size=(0.1, 0.3), color=0, seed=None):
        super().__init__(env)
        self._n, self._size, self._color = int(num_patches), (float(size[0]), float(size[1])), color
        self._rng = np.random.default_rng(seed)
        self._patches: dict[tuple[int, int], list] = {}       # per frame size

    def _sample(self, h, w):
        out = []
        for _ in range(self._n):
            ph = max(1, int(self._rng.uniform(*self._size) * h))
            pw = max(1, int(self._rng.uniform(*self._size) * w))
            y = int(self._rng.integers(0, max(1, h - ph + 1)))
            x = int(self._rng.integers(0, max(1, w - pw + 1)))
            out.append((y, x, ph, pw))
        return out

    def _apply(self, frame):
        h, w = frame.shape[:2]
        if (h, w) not in self._patches:
            self._patches[(h, w)] = self._sample(h, w)
        out = frame.copy()
        for y, x, ph, pw in self._patches[(h, w)]:
            out[y:y + ph, x:x + pw] = self._color
        return out

    def reset(self, **kwargs):
        self._patches = {}
        return super().reset(**kwargs)


class MovingPatchWrapper(_PixelTransform):
    """
    Overlay `num_patches` solid-color rectangles drifting with their own velocity (`speed` pixels per
    step, sampled at reset), reflecting off the frame edges. Patches are kept per frame size.
    """
    def __init__(self, env, num_patches=1, size=(0.1, 0.2), color=255, speed=2.0, seed=None):
        super().__init__(env)
        self._n, self._size, self._color = int(num_patches), (float(size[0]), float(size[1])), color
        self._speed = float(speed)
        self._rng = np.random.default_rng(seed)
        self._patches: dict[tuple[int, int], list] = {}

    def _init_patches(self, h, w):
        out = []
        for _ in range(self._n):
            ph = max(1, int(self._rng.uniform(*self._size) * h))
            pw = max(1, int(self._rng.uniform(*self._size) * w))
            y = float(self._rng.uniform(0, max(1, h - ph)))
            x = float(self._rng.uniform(0, max(1, w - pw)))
            angle = float(self._rng.uniform(0, 2 * math.pi))
            out.append([y, x, ph, pw, self._speed * math.sin(angle), self._speed * math.cos(angle)])
        return out

    def _advance(self):
        for (h, w), patches in self._patches.items():
            for p in patches:
                p[0] += p[4]
                p[1] += p[5]
                ymax, xmax = h - p[2], w - p[3]
                if p[0] < 0:
                    p[0], p[4] = -p[0], -p[4]
                elif p[0] > ymax:
                    p[0], p[4] = 2 * ymax - p[0], -p[4]
                if p[1] < 0:
                    p[1], p[5] = -p[1], -p[5]
                elif p[1] > xmax:
                    p[1], p[5] = 2 * xmax - p[1], -p[5]

    def _apply(self, frame):
        h, w = frame.shape[:2]
        if (h, w) not in self._patches:
            self._patches[(h, w)] = self._init_patches(h, w)
        out = frame.copy()
        for y, x, ph, pw, _, _ in self._patches[(h, w)]:
            yi, xi = int(y), int(x)
            out[yi:yi + ph, xi:xi + pw] = self._color
        return out

    def reset(self, **kwargs):
        self._patches = {}
        return super().reset(**kwargs)

    def step(self, action):
        out = super().step(action)
        self._advance()
        return out


class RandomShiftWrapper(_PixelTransform):
    "DrQ-style random shift: replicate-pad by `pad` pixels, then crop back at random; resampled every frame."
    def __init__(self, env, pad=4, seed=None):
        super().__init__(env)
        self._pad = int(pad)
        self._rng = np.random.default_rng(seed)

    def _apply(self, frame):
        p = self._pad
        padded = cv2.copyMakeBorder(frame, p, p, p, p, cv2.BORDER_REPLICATE)
        dy, dx = int(self._rng.integers(0, 2 * p + 1)), int(self._rng.integers(0, 2 * p + 1))
        h, w = frame.shape[:2]
        return padded[dy:dy + h, dx:dx + w]


class CutoutWrapper(_PixelTransform):
    "Mask `num` random rectangles per frame with `color`; resampled every frame."
    def __init__(self, env, num=1, size=(0.1, 0.2), color=0, seed=None):
        super().__init__(env)
        self._n, self._size, self._color = int(num), (float(size[0]), float(size[1])), color
        self._rng = np.random.default_rng(seed)

    def _apply(self, frame):
        h, w = frame.shape[:2]
        out = frame.copy()
        for _ in range(self._n):
            ch = max(1, int(self._rng.uniform(*self._size) * h))
            cw = max(1, int(self._rng.uniform(*self._size) * w))
            y = int(self._rng.integers(0, max(1, h - ch + 1)))
            x = int(self._rng.integers(0, max(1, w - cw + 1)))
            out[y:y + ch, x:x + cw] = self._color
        return out


class RandomConvWrapper(_PixelTransform):
    "The frame through a randomly initialized 3 -> 3 channel conv; weights resampled at each reset."
    def __init__(self, env, kernel_size=3, seed=None):
        super().__init__(env)
        self._k = int(kernel_size) | 1
        self._rng = np.random.default_rng(seed)
        self._sample()

    def _sample(self):
        scale = 1.0 / (self._k * self._k * 3)
        self._weights = self._rng.normal(0, math.sqrt(scale), (3, self._k, self._k, 3)).astype(np.float32)

    def _apply(self, frame):
        f = frame.astype(np.float32)
        out = np.zeros_like(f)
        for o in range(3):
            for i in range(3):
                out[..., o] += cv2.filter2D(f[..., i], -1, self._weights[o, :, :, i])
        return np.clip(out, 0, 255).astype(frame.dtype)

    def reset(self, **kwargs):
        self._sample()
        return super().reset(**kwargs)


class GrayscaleWrapper(_PixelTransform):
    "Convert to grayscale; `keep_channels=True` broadcasts back to 3 channels."
    def __init__(self, env, keep_channels=True):
        super().__init__(env)
        self._keep = bool(keep_channels)

    def _apply(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        return np.stack([gray] * 3, axis=-1) if self._keep else gray


class ResolutionWrapper(_PixelTransform):
    "Downsample by `scale`, then upsample back to the original size."
    def __init__(self, env, scale=0.5):
        super().__init__(env)
        self._scale = float(scale)

    def _apply(self, frame):
        h, w = frame.shape[:2]
        sw, sh = max(1, int(round(w * self._scale))), max(1, int(round(h * self._scale)))
        small = cv2.resize(frame, (sw, sh), interpolation=cv2.INTER_AREA)
        return cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)

### Tests

In [ ]:
import stable_marl as sm
from functools import partial

ENV = dict(agents=2, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=8)
def views(**world_kwargs):
    world = sm.World('MultiGrid-FindGoal-15x15-v0', num_envs=1, **ENV, **world_kwargs)
    world.reset(seed=0)
    return world.infos['pixels'][0, 0]                                 # (agents, H, W, 3)

clean = views()
for where in ('pre_wrappers', 'extra_wrappers'):                       # before or after MegaWrapper
    gray = views(**{where: [GrayscaleWrapper]})
    assert gray.shape == clean.shape and (gray[..., 0] == gray[..., 1]).all() and not np.array_equal(gray, clean)
    blurred = views(**{where: [partial(BlurWrapper, kernel=5)]})
    assert blurred.shape == clean.shape and not np.array_equal(blurred, clean)

# per-frame randomness: each agent's view its own noise; episode-level samples shared by the agents
noisy = views(extra_wrappers=[partial(NoiseWrapper, std=20.0, seed=0)])
assert not np.array_equal(noisy[0] - clean[0], noisy[1] - clean[1])
world = sm.World('MultiGrid-FindGoal-15x15-v0', num_envs=1, **ENV,
                 extra_wrappers=[partial(OcclusionWrapper, num_patches=2, size=(0.3, 0.4), color=255, seed=0)])
world.reset(seed=0)
occluded = world.infos['pixels'][0, 0]
for y, x, ph, pw in world.envs.envs[0]._patches[occluded.shape[1:3]]:
    assert (occluded[:, y:y + ph, x:x + pw] == 255).all()              # the same patches on every agent's view

# chroma key: the black background replaced by a solid color
keyed = views(extra_wrappers=[partial(ChromaKeyWrapper, key_color=(0, 0, 0), media=np.full((10, 10, 3), 77, np.uint8))])
assert ((clean == 0).all(-1) == (keyed == 77).all(-1)).all()

# schedules, and the render with views of another size (patches per frame size)
assert linear(0, 10, 10)(5) == 5 and linear(0, 10, 10)(50) == 10 and constant(3)(100) == 3
assert abs(cosine(1, 0, 10)(10)) < 1e-9 and exponential(1.0, 0.5, floor=0.2)(10) == 0.2
assert abs(sinusoidal(-1, 1, 4)(1) - 1) < 1e-9
world = sm.World('MultiGrid-FindGoal-15x15-v0', num_envs=1, add_pixels=True, image_shape=(40, 40), **ENV,
                 extra_wrappers=[partial(MovingPatchWrapper, num_patches=1, size=(0.2, 0.3), seed=0)])
world.set_policy(sm.RandomPolicy(seed=0))
world.reset(seed=0)
assert world.infos['render'].shape[2:] != world.infos['pixels'].shape[3:] and len(world.envs.envs[0]._patches) == 2
world.evaluate(episodes=1, seed=0)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()